# Simple decision-tree baseline

This notebook uses the provided operational data while keeping the model deliberately simple: one shallow decision tree. All features are known at the start of a scheduling case or inside its known 168-hour planning window.

## 1. Imports

In [7]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier

from kaggle_metric import score

## 2. Read the training data and feature tables

In [8]:
PROJECT_ROOT = Path.cwd()
KERNEL = PROJECT_ROOT / 'data' / 'kernel'
EXTENDED = PROJECT_ROOT / 'data' / 'extended'

train = pd.read_csv(KERNEL / 'Unit_commitment_decisions.csv')
train['starttime'] = pd.to_datetime(train['starttime'], utc=True).dt.tz_localize(None)
target_columns = [c for c in train.columns if c.startswith('result_committed_')]

def read_time_table(path):
    table = pd.read_csv(path)
    table['date'] = pd.to_datetime(table['date'], utc=True).dt.tz_localize(None)
    return table.set_index('date').sort_index()

price = read_time_table(KERNEL / 'Historical_day_ahead_price_2015_2025.csv')
inflow = read_time_table(KERNEL / 'Historical_inflow_1958_2025.csv')
volume = read_time_table(KERNEL / 'Historical_volume_2015_2024.csv')
water_value = read_time_table(KERNEL / 'Synthetic_water_value_2015_2024.csv')
min_volume = read_time_table(EXTENDED / 'Constraint_min_volume.csv')
min_flow = read_time_table(EXTENDED / 'Constraint_min_flow.csv')

print('Training cases:', len(train))
print('Targets:', len(target_columns))

Training cases: 2922
Targets: 2352


## 3. Build simple features

The model receives all 168 hourly prices directly. Inflow and constraints are summarized over the known week to keep the table small. Storage and water values are taken at the beginning and end of the scheduling window. `Run No` and `total_calculation_time` are not features.

In [9]:
HORIZON = 168

def weekly_window(table, starts):
    offsets = np.arange(HORIZON, dtype='timedelta64[h]')
    timestamps = starts.to_numpy(dtype='datetime64[ns]')[:, None] + offsets
    values = table.reindex(pd.DatetimeIndex(timestamps.ravel())).to_numpy()
    return values.reshape(len(starts), HORIZON, table.shape[1])

def daily_values(table, dates):
    return table.reindex(pd.DatetimeIndex(dates)).to_numpy()

def make_features(cases):
    starts = cases['starttime']
    price_week = weekly_window(price, starts)[:, :, 0]
    inflow_week = weekly_window(inflow, starts)
    constraints_week = np.concatenate([weekly_window(min_volume, starts), weekly_window(min_flow, starts)], axis=2)

    calendar = pd.DataFrame({'month': starts.dt.month, 'day_of_year': starts.dt.dayofyear, 'day_of_week': starts.dt.dayofweek, 'is_weekend': (starts.dt.dayofweek >= 5).astype(int)}, index=cases.index)
    price_features = pd.DataFrame(price_week, index=cases.index, columns=[f'price_t{hour}' for hour in range(HORIZON)])
    inflow_features = pd.DataFrame(
        np.concatenate([inflow_week.mean(axis=1), inflow_week.min(axis=1), inflow_week.max(axis=1)], axis=1),
        index=cases.index,
        columns=[f'inflow_mean_{name}' for name in inflow.columns] + [f'inflow_min_{name}' for name in inflow.columns] + [f'inflow_max_{name}' for name in inflow.columns],
    )
    constraint_names = [*min_volume.columns, *min_flow.columns]
    constraint_features = pd.DataFrame(
        np.concatenate([constraints_week.max(axis=1), (constraints_week > 0).mean(axis=1)], axis=1),
        index=cases.index,
        columns=[f'constraint_max_{name}' for name in constraint_names] + [f'constraint_share_active_{name}' for name in constraint_names],
    )
    volume_features = pd.DataFrame(daily_values(volume, starts), index=cases.index, columns=[f'volume_{name}' for name in volume.columns])
    water_start = pd.DataFrame(daily_values(water_value, starts), index=cases.index, columns=[f'water_start_{name}' for name in water_value.columns])
    water_end = pd.DataFrame(daily_values(water_value, starts + pd.Timedelta(days=7)), index=cases.index, columns=[f'water_end_{name}' for name in water_value.columns])

    features = pd.concat([calendar, price_features, inflow_features, constraint_features, volume_features, water_start, water_end], axis=1)
    assert not features.isna().any().any(), 'Feature table contains missing values'
    return features


## 4. Create inputs and targets

In [10]:
X = make_features(train)
y = train[target_columns]
print('Feature table shape:', X.shape)
X.iloc[:, :8].head()

Feature table shape: (2922, 295)


,month,day_of_year,day_of_week,is_weekend,price_t0,price_t1,price_t2,price_t3
0,1,1,3,0,27.07,26.93,26.83,26.81
1,1,2,4,0,26.41,26.20,25.60,24.74
2,1,3,5,1,25.54,25.43,25.34,25.29
3,1,4,6,1,27.38,27.17,27.03,27.04
4,1,5,0,0,28.15,28.13,28.05,28.04


## 5. Time-based train/validation split

We hold out all of 2022. Training ends seven days before 2022 starts so the rolling weekly windows cannot overlap.

In [11]:
validation_start = pd.Timestamp('2022-01-01')
validation_end = pd.Timestamp('2023-01-01')
train_mask = train['starttime'] < validation_start - pd.Timedelta(days=7)
validation_mask = (train['starttime'] >= validation_start) & (train['starttime'] < validation_end)

X_train, y_train = X.loc[train_mask], y.loc[train_mask]
X_valid, y_valid = X.loc[validation_mask], y.loc[validation_mask]
print('Train cases:', len(X_train))
print('Validation cases:', len(X_valid))

Train cases: 2550
Validation cases: 365


## 6. Train one small decision tree

In [12]:
model = DecisionTreeClassifier(max_depth=5, min_samples_leaf=20, random_state=42)
model.fit(X_train, y_train)

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",5
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",20
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split among considered features for this split.",'best'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: splitting may inspect more than ``max_features`` features ifneeded to find a valid split.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples i

## 7. Evaluate with the official metric

In [13]:
all_probabilities = model.predict_proba(X_valid)
valid_probabilities = np.column_stack([
    probabilities[:, list(model.classes_[column_number]).index(1)] if 1 in model.classes_[column_number] else np.zeros(len(X_valid))
    for column_number, probabilities in enumerate(all_probabilities)
])
valid_submission = pd.DataFrame(valid_probabilities, columns=target_columns)
valid_submission.insert(0, 'Run No', train.loc[validation_mask, 'Run No'].to_numpy())
valid_solution = train.loc[validation_mask, ['Run No', *target_columns]]

auc = score(valid_solution, valid_submission)
print(f'2022 validation micro ROC-AUC: {auc:.4f}')

2022 validation micro ROC-AUC: 0.6736
